<a href="https://colab.research.google.com/github/ChitrashreeShankaranandha/AgenticAI_LLMs_RAG/blob/main/RAG_QuestionAnswering_LLM_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1 - Installation
# !pip -q install langchain langchain-community langchain-openai langchain-text-splitters langchain-chroma gradio bs4

import os

In [ ]:
# 2 - OpenAI Setup
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OpenAIApi")

# 3- Chat Model + Embeddings Model
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
# 4 - Vector Store (Chroma)
from langchain_chroma import Chroma

vector_store = None
retriever = None

# 5 - Indexing (Single Webpage → Load → Split → Store)
# 5.1 Loading Data (single webpage)

from langchain_community.document_loaders import WebBaseLoader

URL = "https://en.wikipedia.org/wiki/Large_language_model"

loader = WebBaseLoader(web_paths=(URL,))
docs = loader.load()

print("Loaded docs:", len(docs))
print("Preview:", docs[0].page_content[:500])

Loaded docs: 1
Preview: 



Large language model - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools





Donate Create account Log in




In [ ]:
# 5.2 Splitting documents (chunking)

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

splits = text_splitter.split_documents(docs)

print("Chunks:", len(splits))
print(splits[0].page_content[:300])

Chunks: 194
Large language model - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload file


In [ ]:
# 5.3 Storing Document (embeddings → vector store)

from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="single_webpage_rag",
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# Quick retrieval sanity check
test = retriever.invoke("What is this page about?")
print("Retrieved:", len(test))
print(test[0].page_content[:300])

Retrieved: 4
This page was last edited on 18 March 2026, at 19:17 (UTC).
Text is available under the Creative Commons Attribution-ShareAlike 4.0 License;
additional terms may apply. By using this site, you agree to the Terms of Use and Privacy Policy. Wikipedia® is a registered trademark of the Wikimedia Foundat


In [ ]:
# 6 - RAG Prompt (answer ONLY from retrieved context)
from langchain_core.prompts import ChatPromptTemplate

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a website assistant. Answer ONLY using the provided context.\n"
     "If the answer is not in the context, say: \"I don't know based on the website.\""),
    ("human",
     "Context:\n{context}\n\nQuestion:\n{question}")
])

In [ ]:
# 7 - Conversational requirement (follow-ups)
# Your “extra requirement” is conversational follow-ups like “the oldest one.”
# To make that work reliably, we do a lightweight 2-step approach:
# Rewrite the user question into a standalone question using chat history
# Retrieve using the rewritten question, then answer with the QA prompt

# 7.1 Question rewrite prompt

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite the user's latest question into a standalone question.\n"
     "Use the chat history for references (pronouns like it/that/oldest).\n"
     "Return ONLY the rewritten standalone question."),
    ("human",
     "Chat history:\n{chat_history}\n\nLatest question:\n{question}")
])

In [ ]:
# 7.2 Core chat function (the “Interactive Chat Function” style)

def format_history(history, max_turns=8):
    history = history[-max_turns:]
    lines = []
    for u, a in history:
        lines.append(f"User: {u}\nAssistant: {a}")
    return "\n\n".join(lines)

def retrieve_context(query: str) -> str:
    docs = retriever.invoke(query)
    if not docs:
        return ""
    return "\n\n".join(d.page_content for d in docs)

def answer_question(user_message: str, history):
    chat_history_text = format_history(history)

    rewritten = llm.invoke(
        rewrite_prompt.format_messages(
            chat_history=chat_history_text,
            question=user_message
        )
    ).content.strip()

    context = retrieve_context(rewritten)

    if not context.strip():
        answer = "I don't know based on the website."
    else:
        answer = llm.invoke(
            qa_prompt.format_messages(
                context=context,
                question=user_message
            )
        ).content.strip()

    new_history = history + [(user_message, answer)]
    return new_history, rewritten

In [ ]:
# 8 - Simple Gradio UI (required deliverable suggestion)

import gradio as gr
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma

with gr.Blocks() as demo:
    gr.Markdown("# Single-Webpage RAG Chatbot")

    url_box = gr.Textbox(value=URL, label="Single webpage URL")
    rebuild_btn = gr.Button("Rebuild Index")
    status = gr.Markdown("")

    chatbot = gr.Chatbot(label="Chat")
    user_box = gr.Textbox(label="Ask a question")
    rewritten_box = gr.Textbox(label="Rewritten question", interactive=False)
    state = gr.State([])   # list of (user, assistant)

    def rebuild(url):
        global vector_store, retriever

        loader = WebBaseLoader(web_paths=(url,))
        docs = loader.load()
        splits = text_splitter.split_documents(docs)

        if not splits:
            return [], [], "No chunks created. Try another URL."

        vector_store = Chroma.from_documents(
            documents=splits,
            embedding=embeddings,
            collection_name="single_webpage_rag",
        )
        retriever = vector_store.as_retriever(search_kwargs={"k": 4})

        return [], [], f"Index rebuilt. Chunks: {len(splits)}"

    rebuild_btn.click(
        rebuild,
        inputs=[url_box],
        outputs=[chatbot, state, status]
    )

    def respond(message, history):
        if retriever is None:
            return history, history, "Click Rebuild Index first.", ""

        new_history, rewritten = answer_question(message, history)
        return new_history, new_history, rewritten, ""

    send_btn = gr.Button("Send")
    send_btn.click(
        respond,
        inputs=[user_box, state],
        outputs=[chatbot, state, rewritten_box, user_box]
    )

demo.launch(debug=True)

/tmp/ipykernel_14565/1512286238.py:114: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat")
/tmp/ipykernel_14565/1512286238.py:114: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9bb4671c71f7136e5b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9bb4671c71f7136e5b.gradio.live
